# `data` -- Data Loading & Team-Name Normalisation

This module is the **entry point of the whole pipeline**: everything downstream (Elo ratings, features, models) consumes the DataFrame this module produces.

It solves one specific, recurring headache in IPL data: **franchises change names and disappear across 18 seasons** (Delhi Daredevils became Delhi Capitals in 2019, Deccan Chargers folded and their 'slot' was taken by Sunrisers Hyderabad, etc.). If we don't normalise these, the same real-world team gets treated as two different teams by Elo/form/head-to-head calculations, which silently corrupts every downstream feature.

Ported from `project_gagan/src/data.py`, with two functions added (`load_ball_by_ball`, `build_match_table`) that were previously inline notebook code in project_gagan's original pipeline notebook.

## Imports

`from __future__ import annotations` lets us write modern type hints like `dict[str, str]` even though this project runs on Python 3.9 (which would otherwise require `Dict[str, str]` from the `typing` module).

`pandas` is the only external dependency -- this module is pure data wrangling, no ML.

In [1]:
import pandas as pd

## `NAME_MAP` -- historical franchise renames

A hand-curated lookup table mapping every team name that has ever changed to its **current** canonical name. Applied everywhere a team name is read from the raw data, so that (for example) a 2015 match played by "Delhi Daredevils" and a 2022 match played by "Delhi Capitals" are treated as the exact same team for Elo/form purposes.

**Defunct franchises are a special case**, marked `DEF-L01`:
- Deccan Chargers folded after 2012; Sunrisers Hyderabad effectively took their IPL slot from 2013 onward, so we map Deccan Chargers's history onto Sunrisers Hyderabad (a judgement call: they're different legal entities, but for a *ratings* system this treats the historical results as belonging to "whoever currently plays in that slot").
- Pune Warriors also folded (after 2013) with no successor franchise, so it maps to itself -- included explicitly so it's clear this was a deliberate decision, not an oversight.

In [2]:
NAME_MAP: dict[str, str] = {
    "Delhi Daredevils":             "Delhi Capitals",
    "Kings XI Punjab":              "Punjab Kings",
    "Royal Challengers Bangalore":  "Royal Challengers Bengaluru",
    "Rising Pune Supergiant":       "Rising Pune Supergiants",
    # defunct franchises — DEF-L01
    "Deccan Chargers":              "Sunrisers Hyderabad",
    "Pune Warriors":                "Pune Warriors",
}

## `ABBREV` -- short-code lookup for the 2026 external dataset

The locked 2026 holdout CSVs (`data/external_2026/*.csv`) use 3-4 letter codes (e.g. `RCB`, `MI`) instead of full team names, because that's how the source scorecards abbreviate them. This dict translates those codes back to the same full names used everywhere else, so the 2026 evaluation can be joined against historical Elo ratings/features without a name mismatch.

In [3]:
ABBREV: dict[str, str] = {
    "SRH":  "Sunrisers Hyderabad",
    "RCB":  "Royal Challengers Bengaluru",
    "KKR":  "Kolkata Knight Riders",
    "MI":   "Mumbai Indians",
    "CSK":  "Chennai Super Kings",
    "RR":   "Rajasthan Royals",
    "GT":   "Gujarat Titans",
    "PBKS": "Punjab Kings",
    "LSG":  "Lucknow Super Giants",
    "DC":   "Delhi Capitals",
}

## `normalise()`

Thin wrapper around `NAME_MAP.get()`. Given any team name, returns the canonical name if it's a known rename, or the name unchanged if it's already canonical (or unrecognised). This is the function `tests/test_features.py` exercises directly (`test_name_map_known`, `test_name_map_deccan_chargers`, `test_name_map_pune_warriors`, `test_name_map_passthrough`).

In [4]:
def normalise(name: str) -> str:
    """Return the canonical team name, mapping old/defunct franchise names."""
    return NAME_MAP.get(name, name)

## `load_ball_by_ball()` -- the real entry point

Reads the raw `ipl_data.xlsx` workbook's **Ball by Ball** sheet (one row per delivery, 273,503 rows across 2008-2025) and applies two cleaning steps:

1. **Team-name normalisation** on every column that holds a team    name (`batting_team`, `bowling_team`, `match_winner`,    `toss_winner`) via `NAME_MAP`.
2. **Filter to standard, completed matches**: keep only innings 1    and 2 (excludes weird edge cases like a canceled/abandoned    match structure), and drop any match with a non-null    `result_type` (this column is populated for D/L-method or    no-result matches -- we don't want those skewing win-rate    calculations).

It also derives `batting_wins`: a simple 1/0 flag for whether the team batting on this delivery went on to win the match. This one column is the foundation every win-probability label in this project is built from.

Ported from `project_gagan` notebook cell 4 (previously inline notebook code, now a reusable, testable function).

In [5]:
def load_ball_by_ball(xlsx_path: str) -> pd.DataFrame:
    """
    Load and clean the 'Ball by Ball' sheet: normalise team names, keep only
    standard completed innings (1/2, no DLS/no-result), and add
    ``batting_wins`` (1 if the batting team won that match).

    Ported from project_gagan notebook cell 4.
    """
    raw = pd.read_excel(xlsx_path, sheet_name="Ball by Ball")

    for col in ["batting_team", "bowling_team", "match_winner", "toss_winner"]:
        if col in raw.columns:
            raw[col] = raw[col].replace(NAME_MAP)

    df = raw[raw["innings"].isin([1, 2]) & raw["result_type"].isna()].copy()
    df["batting_wins"] = (df["batting_team"] == df["match_winner"]).astype(int)
    return df

## `build_match_table()` -- one row per match

`load_ball_by_ball()` gives us one row **per delivery**. All the pre-match/team-level modelling in `pipeline.py` needs one row **per match** instead (team1's Elo, team2's Elo, who won, etc.) -- that's what this function builds.

Key mechanics:
- Groups by `match_id`, taking the **1st-innings batting team as   `team1`** (i.e. `team1` is always the team that batted first,   not a home/away distinction -- this convention is used   consistently by every downstream feature).
- `score1`/`score2` are each innings's final `team_runs` value   (the last ball of that innings).
- `team1_win`: 1 if the team that batted first also won the match   -- this is the primary classification target for the pre-match   model.
- `toss_bat_first` / `toss_field_first`: binary flags for whether   the toss winner chose to bat or field, expressed relative to   `team1`/`team2` so they can be used directly as model features.

Ported from `project_gagan` notebook cell 6.

In [6]:
def build_match_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate ball-by-ball rows into one row per match with team1 (bat-first),
    team2 (bowl-first), scores, toss features, and the team1_win label.

    Ported from project_gagan notebook cell 6.
    """
    match_df = (
        df[df["innings"] == 1]
        .groupby("match_id")
        .agg(
            team1=("batting_team", "first"),
            team2=("bowling_team", "first"),
            winner=("match_winner", "first"),
            year=("year", "first"),
            venue=("venue", "first"),
            toss_winner=("toss_winner", "first"),
            toss_decision=("toss_decision", "first"),
            score1=("team_runs", "last"),
        )
        .reset_index()
        .sort_values("match_id")
        .reset_index(drop=True)
    )
    score2_map = df[df["innings"] == 2].groupby("match_id")["team_runs"].last()
    match_df["score2"] = match_df["match_id"].map(score2_map)
    match_df["team1_win"] = (match_df["team1"] == match_df["winner"]).astype(int)
    match_df["toss_bat_first"] = (
        (match_df["toss_winner"] == match_df["team1"])
        & (match_df["toss_decision"] == "bat")
    ).astype(int)
    match_df["toss_field_first"] = (
        (match_df["toss_winner"] == match_df["team2"])
        & (match_df["toss_decision"] == "field")
    ).astype(int)
    return match_df